# 🎮 Steam Nexus: Data Engineering Pipeline (Games)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PATH_GAMES = "../data/raw/steam_games_raw.csv"
headers_fixed = [
    'AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age',
    'Price', 'DiscountDLC count', 'About the game', 'Supported languages',
    'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url',
    'Support email', 'Extra_Column', 'Windows', 'Mac', 'Linux', 'Metacritic score',
    'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank',
    'Achievements', 'Recommendations', 'Notes', 'Average playtime forever',
    'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks',
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies'
]

df_games = pd.read_csv(PATH_GAMES, low_memory=False, names=headers_fixed, skiprows=1)
print("✅ Dataset loaded for analysis.")

## 🔍 1. Structural Recognition
Identifying the dataset magnitude and the initial state of data types.

In [ ]:
print(f"📏 Dimensions: {df_games.shape[0]} rows x {df_games.shape[1]} columns")
display(df_games.head(3))

In [ ]:
print("\n--- Initial Typing ---")
df_games.info()

In [ ]:
df_games.isnull().sum()

In [ ]:
print("\n--- Memory Estimation ---")
mem_usage = df_games.memory_usage(deep=True).sum() / (1024**2)
print(f"Total memory used by DataFrame: {mem_usage:.2f} MB")

### 1.2 Scale and Composition Analysis
Extending our structural recognition to understand the granularity and dispersion of our data through unique counts and descriptive statistics.

In [ ]:
print("\n--- Unique Values Count (Selected Columns) ---")
unique_counts = {
    'AppID': df_games['AppID'].nunique(),
    'Name': df_games['Name'].nunique(),
    'Release date': df_games['Release date'].nunique(),
    'Developers': df_games['Developers'].nunique(),
    'Publishers': df_games['Publishers'].nunique(),
    'Categories': df_games['Categories'].nunique(),
    'Genres': df_games['Genres'].nunique(),
    'Tags': df_games['Tags'].nunique(),
    'Supported languages': df_games['Supported languages'].nunique()
}
for col, count in unique_counts.items():
    print(f"'{col}': {count} unique values")

In [ ]:
print("\n--- Descriptive Statistics (Key Numerical Columns) ---")
df_games_numeric_cols = [
    'Price', 'Estimated owners', 'Peak CCU', 'Required age',
    'Metacritic score', 'Positive', 'Negative', 'Achievements', 'Recommendations',
    'Average playtime forever', 'Median playtime forever'
]

# Convert relevant columns to numeric, coercing errors to NaN
for col in df_games_numeric_cols:
    if col in df_games.columns:
        df_games[col] = pd.to_numeric(df_games[col], errors='coerce')

display(df_games[df_games_numeric_cols].describe().transpose())

In [ ]:
print("\n--- 'Extra_Column' Verification ---")
if 'Extra_Column' in df_games.columns:
    if df_games['Extra_Column'].isnull().all():
        print("'Extra_Column' is completely empty and can be removed.")
        # df_games = df_games.drop(columns=['Extra_Column']) # Uncomment to remove
    else:
        print("'Extra_Column' contains data. First 5 values:\n", df_games['Extra_Column'].head())
else:
    print("'Extra_Column' not found in DataFrame.")

## 🎯 Step 2: Hypothesis and Visualization Based Justification

In this section, we visually demonstrate why we selected each critical variable for our data product.

### 2.1 `Price` Justification (Economic Variable)
**Hypothesis:** Price is not uniform on Steam; there is a clear segmentation between free and premium games that will affect the recommendation system.

**Visual Evidence:**

In [ ]:
df_games['Price_num'] = pd.to_numeric(df_games['Price'], errors='coerce').fillna(0.0)
plt.figure(figsize=(10, 5))
sns.histplot(df_games[df_games['Price_num'] < 60]['Price_num'], bins=30, kde=True, color='teal')
plt.title("Price Distribution: Market Segmentation")
plt.xlabel("Price in USD")
plt.show()
print("Conclusion: High variability justifies using Price as a segmentation variable.")

### 2.2 `Genres` / `Tags` Justification (Recommendation Engine)
**Hypothesis:** Most games share multiple tags, allowing for the creation of a co-occurrence network for graph analysis.

**Visual Evidence:**

In [ ]:
top_genres = df_games['Genres'].str.split(',').explode().str.strip().value_counts().head(10)
plt.figure(figsize=(12, 5))
top_genres.plot(kind='bar', color='coral')
plt.title("Top 10 Genres: Connectivity Potential")
plt.ylabel("Frequency")
plt.show()
print("Conclusion: Genre richness justifies the use of Graphs and Clustering.")

### 2.3 `Positive` / `Negative` Justification (Success Metrics)
**Hypothesis:** The volume of positive reviews correlates with popularity, allowing for the classification of successful games.

**Visual Evidence:**

In [ ]:
df_games['Positive'] = pd.to_numeric(df_games['Positive'], errors='coerce').fillna(0)
df_games['Negative'] = pd.to_numeric(df_games['Negative'], errors='coerce').fillna(0)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Positive', y='Negative', data=df_games[df_games['Positive'] < 50000], alpha=0.5)
plt.title("Review Correlation: Identifying Commercial Success")
plt.show()
print("Conclusion: These metrics will allow us to label 'success' for predictive models.")

### 2.4 `Name` and `About the game` Justification (Identification and Content)
**Hypothesis:** The name is a key identifier for each game, and its description (About the game) offers valuable textual content for natural language processing models and content-based game recommendations. Cleaning these variables is crucial to ensure their consistency and utility.

**Visual Evidence:**

### 2.5 `Developers` Justification (Development Impact and Trends)
**Hypothesis:** Developer identity is a key factor in the quantity of games produced and can influence the type and quality of games released, allowing for the identification of prolific or specialized developers and the analysis of industry trends. This is fundamental for understanding the game ecosystem and for recommendation systems that group games by creator.

In [ ]:
# Count number of games per developer (excluding 'unknown')
top_developers = df_game_work[df_game_work['Developers'] != 'unknown']['Developers'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_developers.index, y=top_developers.values, palette='plasma')
plt.title('Top 10 Developers by Number of Games')
plt.xlabel('Developer')
plt.ylabel('Number of Games')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Conclusion: The existence of developers with multiple titles justifies analyzing their impact and creating profiles for recommendations or market analysis.")

In [ ]:
# Verify name uniqueness (after initial NaN cleaning in Name)
name_duplicates = df_games[df_games['Name'].notnull()].duplicated(subset=['Name']).sum()
print(f"Games with duplicate names (before cleaning): {name_duplicates}")

# Analyze description lengths
df_games['About the game_length'] = df_games['About the game'].fillna('').apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df_games['About the game_length'], bins=50, kde=True, color='purple')
plt.title("Description Length Distribution ('About the game')")
plt.xlabel("Description Length")
plt.ylabel("Frequency")
plt.show()

print("Conclusion: Variable description length indicates information richness, useful for NLP. Name uniqueness is vital.")

## 🛠️ Step 3: Cleaning and Working Table Creation
After justifying the variables, we generate the final clean dataset.

In [ ]:
columns_selected = ['AppID', 'Name', 'Price', 'Genres', 'Tags', 'Positive', 'Negative', 'Release date', 'About the game', 'Developers']
df_game_work = df_games[columns_selected].copy()

In [ ]:
df_game_work.head()

In [ ]:
print("--- 🚩 Empty and Duplicate Detection ---")
empty_names = df_game_work['Name'].isnull().sum()
duplicates = df_game_work.duplicated(subset=['AppID']).sum()
print(f"Rows without name: {empty_names}")
print(f"Duplicate AppIDs: {duplicates}")

In [ ]:
df_game_work.isnull().sum()

In [ ]:
df_game_work.shape

In [ ]:
df_game_work = df_game_work.dropna(subset=['Name'])
df_game_work = df_game_work.drop_duplicates(subset=['AppID'])

In [ ]:
print("\n--- 🔄 Type Normalization and Sanitization ---")

# 1. Price, Positive, and Negative Formatting
df_game_work['Price'] = pd.to_numeric(df_game_work['Price'], errors='coerce').fillna(0.0)
df_game_work['Positive'] = pd.to_numeric(df_game_work['Positive'], errors='coerce').fillna(0)
df_game_work['Negative'] = pd.to_numeric(df_game_work['Negative'], errors='coerce').fillna(0)

# 2. Date Formatting
df_game_work['Release date'] = pd.to_datetime(df_game_work['Release date'], errors='coerce')

# 3. Text Sanitization (Ensuring cast to string before .str)
df_game_work['Name'] = df_game_work['Name'].str.strip().str.title()
df_game_work['Genres'] = df_game_work['Genres'].fillna("Other").str.strip().str.lower()
df_game_work['Tags'] = df_game_work['Tags'].fillna("[]").str.strip().str.lower()
df_game_work['Developers'] = df_game_work['Developers'].fillna("Unknown").str.strip().str.lower()
df_game_work['About the game'] = df_game_work['About the game'].fillna('').str.strip()

In [ ]:
print("✅ Technical cleaning completed.")
df_game_work.shape.head()

In [ ]:
output_path = "../data/processed/steam_games_cleaned_v1.csv"
df_game_work.to_csv(output_path, index=False)
print(f"🎉 Success! The clean dataset has been generated.")
print(f"📍 Location: {output_path}")
print(f"📊 Final records ready: {len(df_game_work)}")